In [10]:
train_sets = ['../data/raw/sample_23.csv'] # do not touch
test_set = '../data/raw/sample_24.csv' # do not touch
max_rows = -1 # -1 for the whole file

# Insert below paths to your clusterings (format required as a CSV: ID,shapelet_cluster)
clusterings = ['outputs/shapelet_experiments/shapelet_cluster_labels.csv', 
                'outputs/kshape/kshape_cluster_labels_sample_23.csv',
                'outputs/kshape/kshape_cluster_labels_sample_23_znorm.csv',
                'outputs/kMedoid/kMedoid_cluster_labels_sample_23.csv',
                'outputs/kMedoid/kMedoid_cluster_labels_sample_23_znorm.csv'
            ] 

In [11]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [12]:
# Helper functions
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["dow"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["day"] = out["date"].dt.day
    out["dayofyear"] = out["date"].dt.dayofyear

    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7.0)
    return out


def add_lag_features(df: pd.DataFrame, group_col: str, target_col: str) -> pd.DataFrame:
    out = df.copy()
    g = out.groupby(group_col)[target_col]

    out["lag_1"] = g.shift(1)
    out["lag_7"] = g.shift(7)
    out["lag_14"] = g.shift(14)
    out["lag_28"] = g.shift(28)

    out["roll_mean_7"] = g.shift(1).rolling(7).mean().reset_index(level=0, drop=True)
    out["roll_mean_28"] = g.shift(1).rolling(28).mean().reset_index(level=0, drop=True)
    return out


def build_cluster_mapping(clustering_df):
    clustering_df = clustering_df.copy()
    clustering_df.columns = clustering_df.columns.str.strip()

    id_candidates = [c for c in clustering_df.columns if c.lower() == "id"]
    cluster_candidates = [c for c in clustering_df.columns if "cluster" in c.lower()]

    cluster_id_col = id_candidates[0] if id_candidates else clustering_df.columns[0]
    cluster_col = cluster_candidates[0] if cluster_candidates else clustering_df.columns[1]

    cluster_map = clustering_df[[cluster_id_col, cluster_col]].dropna().drop_duplicates().copy()
    cluster_map[cluster_id_col] = cluster_map[cluster_id_col].astype(str)

    cluster_values = sorted(pd.unique(cluster_map[cluster_col]))
    cluster_value_to_code = {v: i for i, v in enumerate(cluster_values)}

    id_to_cluster_value = dict(zip(cluster_map[cluster_id_col], cluster_map[cluster_col]))
    id_to_cluster_code = {
        item_id: int(cluster_value_to_code[value])
        for item_id, value in id_to_cluster_value.items()
    }

    return cluster_map, cluster_id_col, cluster_col, id_to_cluster_value, id_to_cluster_code


def train_model_with_cluster_feature(train_wide_in, id_col, id_to_cluster_code):
    def add_cluster_code_feature(df):
        out = df.copy()
        out["cluster_code"] = out[id_col].astype(str).map(id_to_cluster_code).fillna(-1).astype(float)
        return out

    train_model_fn = globals()["train_model"]
    return train_model_fn(
        train_wide_in,
        id_col,
        extra_feature_builder=add_cluster_code_feature,
        extra_feature_cols=["cluster_code"],
    )

In [13]:
def preprocess_test_data(test_wide, id_col):
    test_cols = pd.Index(test_wide.columns)
    test_parsed = pd.to_datetime(test_cols, format="%Y-%m-%d", errors="coerce")
    test_date_mask = (test_cols != id_col) & test_parsed.notna()
    test_date_cols = test_cols[test_date_mask].tolist()
    future_dates = np.array(sorted(pd.to_datetime(test_date_cols)))

    test_long_actual = test_wide.melt(id_vars=id_col, value_vars=test_date_cols, var_name="date", value_name="y_true")
    test_long_actual["date"] = pd.to_datetime(test_long_actual["date"])
    return future_dates, test_long_actual

In [14]:
def train_model(train_wide, id_col, extra_feature_builder=None, extra_feature_cols=None):
    train_cols = pd.Index(train_wide.columns)
    train_parsed = pd.to_datetime(train_cols, format="%Y-%m-%d", errors="coerce")
    train_date_mask = (train_cols != id_col) & train_parsed.notna()
    train_date_cols = train_cols[train_date_mask].tolist()

    train_long = train_wide.melt(id_vars=id_col, value_vars=train_date_cols, var_name="date", value_name="y")
    train_long["date"] = pd.to_datetime(train_long["date"])
    train_long = train_long.sort_values([id_col, "date"]).reset_index(drop=True)

    print("Adding features")
    train_feat = add_calendar_features(train_long)
    train_feat = add_lag_features(train_feat, group_col=id_col, target_col="y")

    if extra_feature_builder is not None:
        train_feat = extra_feature_builder(train_feat)

    feature_cols = [
        "dow", "month", "day", "dayofyear",
        "doy_sin", "doy_cos", "dow_sin", "dow_cos",
        "lag_1", "lag_7", "lag_14", "lag_28",
        "roll_mean_7", "roll_mean_28",
    ]
    if extra_feature_cols:
        feature_cols.extend(extra_feature_cols)

    train_model_df = train_feat.dropna(subset=feature_cols + ["y"]).copy()
    X_train = train_model_df[feature_cols].to_numpy(dtype=float)
    y_train = train_model_df["y"].to_numpy(dtype=float)

    # Model
    print("Fitting model")
    model = HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.05,
        max_iter=300,
        random_state=42,
    )
    model.fit(X_train, y_train)

    return model, feature_cols

In [15]:
def predict(model, future_dates, id_values, train_wide, id_col, feature_cols, extra_feature_builder=None):
    n_ids = len(id_values)
    n_steps = len(future_dates)
    pred_matrix = np.empty((n_ids, n_steps), dtype=float)

    train_date_cols = [
        c for c in train_wide.columns
        if c != id_col and pd.notna(pd.to_datetime(c, format="%Y-%m-%d", errors="coerce"))
    ]

    if len(train_date_cols) == 0:
        raise ValueError("No valid date columns found in train data.")

    train_wide_indexed = train_wide.set_index(id_col)

    history_by_id = {}
    for item_id in id_values:
        if item_id in train_wide_indexed.index:
            hist_series = train_wide_indexed.loc[item_id, train_date_cols]
            history_by_id[item_id] = np.asarray(hist_series, dtype=float).tolist()
        else:
            history_by_id[item_id] = []

    print("Predicting...")

    for step_idx, dt in enumerate(future_dates):
        dow = dt.dayofweek
        month = dt.month
        day = dt.day
        dayofyear = dt.dayofyear

        doy_sin = np.sin(2 * np.pi * dayofyear / 366.0)
        doy_cos = np.cos(2 * np.pi * dayofyear / 366.0)
        dow_sin = np.sin(2 * np.pi * dow / 7.0)
        dow_cos = np.cos(2 * np.pi * dow / 7.0)

        X_batch = np.empty((n_ids, len(feature_cols)), dtype=float)

        for i, item_id in enumerate(id_values):
            hist = history_by_id[item_id]

            lag_1 = hist[-1] if len(hist) >= 1 else np.nan
            lag_7 = hist[-7] if len(hist) >= 7 else lag_1
            lag_14 = hist[-14] if len(hist) >= 14 else lag_1
            lag_28 = hist[-28] if len(hist) >= 28 else lag_1

            roll_mean_7 = np.mean(hist[-7:]) if len(hist) >= 7 else lag_1
            roll_mean_28 = np.mean(hist[-28:]) if len(hist) >= 28 else lag_1

            row_features = [
                dow, month, day, dayofyear,
                doy_sin, doy_cos, dow_sin, dow_cos,
                lag_1, lag_7, lag_14, lag_28,
                roll_mean_7, roll_mean_28,
            ]

            if extra_feature_builder is not None:
                extra_features = extra_feature_builder(item_id=item_id, dt=dt, hist=hist)
                row_features.extend(extra_features)

            X_batch[i, :] = row_features

        y_pred_batch = model.predict(X_batch)
        pred_matrix[:, step_idx] = y_pred_batch

        for i, item_id in enumerate(id_values):
            history_by_id[item_id].append(float(y_pred_batch[i]))

    pred_long = pd.DataFrame(
        {
            id_col: np.repeat(id_values, n_steps),
            "date": np.tile(future_dates, n_ids),
            "y_pred": pred_matrix.reshape(-1),
        }
    )

    return pred_long


def predict_with_cluster_feature(model, future_dates, id_values, train_wide_in, id_col, feature_cols, id_to_cluster_code):
    def add_cluster_feature(item_id, dt, hist):
        _ = (dt, hist)
        return [float(id_to_cluster_code.get(item_id, -1))]

    return predict(
        model,
        future_dates,
        id_values,
        train_wide_in,
        id_col,
        feature_cols,
        extra_feature_builder=add_cluster_feature,
    )


In [16]:
def evaluate(pred_long, train_long, test_long_actual, id_col, pred_out_path="../data/processed/sklearn_forecast_2024.csv"):
    eval_df = pred_long.merge(test_long_actual, on=[id_col, "date"], how="left")

    valid_mask = eval_df["y_true"].notna()

    metrics = {
        "mae": np.nan,
        "rmse": np.nan,
        "nrmse_mean": np.nan,
        "cvrmse_pct": np.nan,
        "nrmse_range": np.nan,
        "n_eval_points": int(valid_mask.sum()),
    }

    if valid_mask.any():
        y_true = eval_df.loc[valid_mask, "y_true"]
        y_pred = eval_df.loc[valid_mask, "y_pred"]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        mean_true = float(y_true.mean())
        y_true_range = float(y_true.max() - y_true.min())

        nrmse_mean = (rmse / mean_true) if abs(mean_true) > 1e-12 else np.nan
        cvrmse_pct = 100.0 * nrmse_mean if np.isfinite(nrmse_mean) else np.nan
        nrmse_range = (rmse / y_true_range) if y_true_range > 1e-12 else np.nan

        metrics.update(
            {
                "mae": float(mae),
                "rmse": float(rmse),
                "nrmse_mean": float(nrmse_mean) if np.isfinite(nrmse_mean) else np.nan,
                "cvrmse_pct": float(cvrmse_pct) if np.isfinite(cvrmse_pct) else np.nan,
                "nrmse_range": float(nrmse_range) if np.isfinite(nrmse_range) else np.nan,
            }
        )

    pred_wide = pred_long.copy()
    pred_wide["date"] = pred_wide["date"].dt.strftime("%Y-%m-%d")
    pred_wide = pred_wide.pivot(index=id_col, columns="date", values="y_pred").reset_index()
    pred_wide.to_csv(pred_out_path, index=False)

    metrics["pred_out_path"] = pred_out_path
    metrics["n_pred_rows"] = int(len(pred_long))

    return metrics, pred_wide, eval_df


def print_evaluation_summary(results):
    if len(results) == 0:
        print("No evaluation results to summarize.")
        return

    summary_df = pd.DataFrame(results)

    cols = [
        "clustering_used",
        "label",
        "n_eval_points",
        "mae",
        "rmse",
        "cvrmse_pct",
        "nrmse_range",
        "pred_out_path",
    ]
    cols = [c for c in cols if c in summary_df.columns]

    print("Evaluation Summary")
    print(summary_df[cols].to_string(index=False))

    numeric_cols = ["mae", "rmse", "cvrmse_pct", "nrmse_range"]
    available_numeric = [c for c in numeric_cols if c in summary_df.columns]

    if len(available_numeric) > 0:
        print("\nAverages")
        print(summary_df[available_numeric].mean(numeric_only=True).to_string())

In [17]:
def run(train_data, test_data, id_col, label="run", pred_out_path="../data/processed/sklearn_forecast_2024.csv"):
    train_data = train_data.copy()
    test_data = test_data.copy()

    # Normalize incoming column names and requested id column to avoid whitespace mismatches.
    train_data.columns = train_data.columns.str.strip()
    test_data.columns = test_data.columns.str.strip()
    id_col = id_col.strip()

    if id_col not in train_data.columns:
        raise ValueError(f"ID column '{id_col}' not found in train data. Available: {list(train_data.columns[:5])}...")
    if id_col not in test_data.columns:
        raise ValueError(f"ID column '{id_col}' not found in test data. Available: {list(test_data.columns[:5])}...")

    model, feature_cols = train_model(train_data, id_col)

    id_values = np.array(sorted(pd.unique(test_data[id_col].dropna())))
    if len(id_values) == 0:
        raise ValueError("No IDs found in test data.")

    future_dates, test_long_actual = preprocess_test_data(test_data, id_col)

    pred_long = predict(model, future_dates, id_values, train_data, id_col, feature_cols)

    metrics, pred_wide, eval_df = evaluate(
        pred_long,
        train_data.melt(id_vars=id_col, var_name="date", value_name="y"),
        test_long_actual,
        id_col,
        pred_out_path=pred_out_path,
    )
    metrics["label"] = label

    return metrics, pred_wide, eval_df

In [18]:
# Data loading
train_path = train_sets[0] if isinstance(train_sets, (list, tuple)) else train_sets
test_path = test_set

if max_rows != -1:
    train_wide = pd.read_csv(train_path, nrows=max_rows)
    test_wide = pd.read_csv(test_path, nrows=max_rows)
else:
    train_wide = pd.read_csv(train_path)
    test_wide = pd.read_csv(test_path)

results = []

print("Running without clusters")
base_metrics, _, _ = run(
    train_wide,
    test_wide,
    id_col="ID",
    label="all_data",
    pred_out_path="../data/processed/sklearn_forecast_2024_all.csv",
)
base_metrics["clustering_used"] = "none"
results.append(base_metrics)

train_norm = train_wide.copy()
train_norm.columns = train_norm.columns.str.strip()
train_norm["ID"] = train_norm["ID"].astype(str)

test_norm = test_wide.copy()
test_norm.columns = test_norm.columns.str.strip()
test_norm["ID"] = test_norm["ID"].astype(str)


def build_train_long(train_wide_in, id_col):
    train_cols = pd.Index(train_wide_in.columns)
    train_date_mask = (train_cols != id_col) & pd.to_datetime(
        train_cols, format="%Y-%m-%d", errors="coerce"
    ).notna()
    train_date_cols = train_cols[train_date_mask].tolist()
    train_long = train_wide_in.melt(
        id_vars=id_col, value_vars=train_date_cols, var_name="date", value_name="y"
    )
    train_long["date"] = pd.to_datetime(train_long["date"])
    return train_long


def evaluate_and_record(pred_long, train_wide_in, test_wide_in, id_col, out_path, label, clustering_used):
    _, test_long_actual = preprocess_test_data(test_wide_in, id_col)
    train_long = build_train_long(train_wide_in, id_col)

    metrics, _, _ = evaluate(
        pred_long,
        train_long,
        test_long_actual,
        id_col,
        pred_out_path=out_path,
    )
    metrics["label"] = label
    metrics["clustering_used"] = clustering_used
    results.append(metrics)


def evaluate_cluster_specific_combined(train_norm_in, test_norm_in, cluster_map, cluster_id_col, cluster_col, out_path, clustering_used):
    cluster_pred_wide_parts = []

    for _, group in cluster_map.groupby(cluster_col):
        cluster_value = group[cluster_col].iloc[0]
        cluster_ids = set(group[cluster_id_col].tolist())

        train_cluster = train_norm_in[train_norm_in["ID"].isin(cluster_ids)].copy()
        test_cluster = test_norm_in[test_norm_in["ID"].isin(cluster_ids)].copy()

        if len(train_cluster) == 0 or len(test_cluster) == 0:
            continue

        cluster_metrics, cluster_pred_wide, _ = run(
            train_cluster,
            test_cluster,
            id_col="ID",
            label=f"cluster_specific_train_cluster_{cluster_value}",
            pred_out_path=f"../data/processed/sklearn_forecast_2024_cluster_specific_cluster_{cluster_value}.csv",
        )
        cluster_metrics["clustering_used"] = clustering_used
        results.append(cluster_metrics)
        cluster_pred_wide_parts.append(cluster_pred_wide)

    if len(cluster_pred_wide_parts) == 0:
        return

    combined_pred_wide = pd.concat(cluster_pred_wide_parts, ignore_index=True)
    combined_pred_wide = combined_pred_wide.drop_duplicates(subset=["ID"], keep="last")
    combined_pred_long = combined_pred_wide.melt(
        id_vars="ID", var_name="date", value_name="y_pred"
    )
    combined_pred_long["date"] = pd.to_datetime(combined_pred_long["date"])

    evaluate_and_record(
        combined_pred_long,
        train_norm_in,
        test_norm_in,
        "ID",
        out_path,
        "cluster_specific_train_combined",
        clustering_used,
    )


def evaluate_global_with_cluster_feature(train_norm_in, test_norm_in, id_to_cluster_code, out_path, label, clustering_used):
    model_cf, feature_cols_cf = train_model_with_cluster_feature(train_norm_in, "ID", id_to_cluster_code)

    id_values = np.array(sorted(pd.unique(test_norm_in["ID"].dropna())))
    future_dates, _ = preprocess_test_data(test_norm_in, "ID")

    pred_long_cf = predict_with_cluster_feature(
        model_cf,
        future_dates,
        id_values,
        train_norm_in,
        "ID",
        feature_cols_cf,
        id_to_cluster_code,
    )

    evaluate_and_record(
        pred_long_cf,
        train_norm_in,
        test_norm_in,
        "ID",
        out_path,
        label,
        clustering_used,
    )


for clustering_path in clusterings:
    print("## CLUSTERING FILE:", clustering_path)
    clustering_used = clustering_path.split('/')[-1].replace('.csv', '')
    clustering_df = pd.read_csv(clustering_path)

    (
        cluster_map,
        cluster_id_col,
        cluster_col,
        id_to_cluster_value,
        id_to_cluster_code,
    ) = build_cluster_mapping(clustering_df)

    cluster_pred_wide_parts = []

    for _, group in cluster_map.groupby(cluster_col):
        cluster_value = group[cluster_col].iloc[0]
        cluster_ids = set(group[cluster_id_col].tolist())
        test_cluster = test_norm[test_norm["ID"].isin(cluster_ids)].copy()
        if len(test_cluster) == 0:
            continue

        cluster_metrics, cluster_pred_wide, _ = run(
            train_norm,
            test_cluster,
            id_col="ID",
            label=f"global_train_cluster_{cluster_value}",
            pred_out_path=f"../data/processed/sklearn_forecast_2024_cluster_{cluster_value}.csv",
        )
        cluster_metrics["clustering_used"] = clustering_used
        results.append(cluster_metrics)
        cluster_pred_wide_parts.append(cluster_pred_wide)

    if len(cluster_pred_wide_parts) > 0:
        combined_pred_wide = pd.concat(cluster_pred_wide_parts, ignore_index=True)
        combined_pred_wide = combined_pred_wide.drop_duplicates(subset=["ID"], keep="last")
        combined_pred_long = combined_pred_wide.melt(
            id_vars="ID", var_name="date", value_name="y_pred"
        )
        combined_pred_long["date"] = pd.to_datetime(combined_pred_long["date"])

        evaluate_and_record(
            combined_pred_long,
            train_norm,
            test_norm,
            "ID",
            f"../data/processed/sklearn_forecast_2024_cluster_combined_{clustering_path.split('/')[-1].replace('.csv', '')}.csv",
            "global_train_cluster_combined",
            clustering_used,
        )

    evaluate_cluster_specific_combined(
        train_norm,
        test_norm,
        cluster_map,
        cluster_id_col,
        cluster_col,
        out_path=f"../data/processed/sklearn_forecast_2024_cluster_specific_combined_{clustering_path.split('/')[-1].replace('.csv', '')}.csv",
        clustering_used=clustering_used,
    )

    evaluate_global_with_cluster_feature(
        train_norm,
        test_norm,
        id_to_cluster_code,
        out_path=f"../data/processed/sklearn_forecast_2024_global_with_cluster_feature_{clustering_path.split('/')[-1].replace('.csv', '')}.csv",
        label="global_with_cluster_feature",
        clustering_used=clustering_used,
    )

print_evaluation_summary(results)


Running without clusters
Adding features
Fitting model
Predicting...
## CLUSTERING FILE: outputs/shapelet_experiments/shapelet_cluster_labels.csv
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
## CLUSTERING FILE: outputs/kshape/kshape_cluster_labels_sample_23.csv
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predicting...
Adding features
Fitting model
Predi